Authentication step

In [2]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")
pid = "snproject-cbb9ca25"

Authenticated


# **COUNTRY ENRICHMENT**

##AUTHORS DATA

### ANALYSIS

###QUERY A

In [ ]:
%%bigquery  --project $pid

WITH cdp_base AS (
  SELECT
    LOWER(email) AS email,
    NULLIF(TRIM(country), '') AS country_clean
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE
    (mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE)
    AND email IS NOT NULL
),

authors_clean AS (
  SELECT
    LOWER(author_email) AS email,
    NULLIF(TRIM(author_country), '') AS author_country_clean
  FROM `datasn-submissions-55677263.prod_joint_submissions_core.authors`
  WHERE author_email IS NOT NULL
),

authors_dedup AS (
  SELECT
    email,
    ARRAY_AGG(author_country_clean ORDER BY cnt DESC, author_country_clean ASC LIMIT 1)[OFFSET(0)]
      AS author_country
  FROM (
    SELECT
      email,
      author_country_clean,
      COUNT(*) AS cnt
    FROM authors_clean
    WHERE author_country_clean IS NOT NULL
    GROUP BY 1,2
  )
  GROUP BY email
),

matched_unique AS (
  SELECT
    c.email,
    c.country_clean,
    a.author_country
  FROM cdp_base c
  INNER JOIN authors_dedup a
    ON c.email = a.email
)

SELECT
  COUNT(*) AS matched_unique_emails,
  COUNTIF(country_clean IS NULL AND author_country IS NOT NULL) AS enrichable_unique_emails
FROM matched_unique;


Query is running:   0%|          |

Downloading:   0%|          |

,matched_unique_emails,enrichable_unique_emails
0,14174968,9304


The CDP dataset is effectively email-unique, and approximately 9.3K records can be reliably enriched with country information from the Authors dataset.

## CRP DATA

###QUERY A

In [ ]:


%%bigquery  --project $pid


WITH cdp_data AS (
  -- 1. CDP (Audience Builder) all ids + canonical id
  SELECT
    canonical_segment_id,
    researcher_id,
    orcid,
    snid,
    country AS cdp_country
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
     OR mkt_pref_opt_in = TRUE
),

crp_rid AS (
  SELECT researcher_id, current_grid_country AS country_rid
  FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`
  WHERE researcher_id IS NOT NULL
    AND current_grid_country IS NOT NULL
),

crp_snid AS (
  SELECT s.snid AS single_snid, current_grid_country AS country_snid
  FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`,
       UNNEST(snids) AS s
  WHERE current_grid_country IS NOT NULL
)


SELECT
  (a.researcher_id IS NOT NULL OR a.snid IS NOT NULL) AS cdp_id_exists,
  (rid.researcher_id IS NOT NULL
    OR sn.single_snid IS NOT NULL ) AS crp_match_exists,
  a.cdp_country IS NOT NULL AS cdp_country_given,
  COALESCE(rid.country_rid, sn.country_snid) IS NOT NULL AS crp_country_given,

  COUNT(DISTINCT a.canonical_segment_id) AS record_count

FROM cdp_data AS a
LEFT JOIN crp_rid  AS rid ON a.researcher_id = rid.researcher_id
LEFT JOIN crp_snid AS sn  ON a.snid = sn.single_snid

GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4;

Query is running:   0%|          |

Downloading:   0%|          |

,cdp_id_exists,crp_match_exists,cdp_country_given,crp_country_given,record_count
0,False,False,False,False,35867
1,False,False,True,False,7545965
2,True,False,False,False,1142380
3,True,False,True,False,2427737
4,True,True,False,True,74002
5,True,True,True,True,4580953


Using CRP data, 74,002 unique CDP records can be reliably enriched with country information where the field is currently missing in CDP.

###QUERY B: Checking the country consistency between CRP and CDP

In [ ]:
%%bigquery  --project $pid

WITH
cdp_data AS
(
  SELECT email as journal_email, country as cdp_country, researcher_id_flag, researcher_id
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = true or mkt_pref_opt_in = true
),
crp_data AS
(
  SELECT researcher_id, current_grid_country as crp_country
  FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`
)

SELECT cdp_country = crp_country as consistent, COUNT(*)
FROM
(
SELECT a.researcher_id_flag as cdp_id, b.researcher_id IS NOT NULL as crp_id, cdp_country is not null as cdp_given, crp_country is not null crp_given, cdp_country, crp_country
FROM cdp_data as a
LEFT JOIN crp_data as b ON a.researcher_id = b.researcher_id
)
WHERE cdp_id = true and crp_id = true and cdp_given = true and crp_given = true
GROUP BY 1





Query is running:   0%|          |

Downloading:   0%|          |

,consistent,f0_
0,False,322431
1,True,4015883


###QUERY C: EXAMPLES TO INCONSISTENCIES

In [ ]:
%%bigquery  --project $pid

WITH
cdp_data AS
(
  SELECT
    email AS journal_email,
    country AS journal_country,
    researcher_id_flag,
    orcid_flag,
    researcher_id
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
),

crp_data AS
(
  SELECT
    researcher_id,
    current_grid_country AS crp_country
  FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`
)

SELECT
  journal_country,
  crp_country,
  COUNT(*) AS cnt
FROM
(
  SELECT
    a.researcher_id_flag AS cdp_id,
    b.researcher_id IS NOT NULL AS crp_id,
    a.journal_country IS NOT NULL AS cdp_given,
    b.crp_country IS NOT NULL AS crp_given,
    a.journal_country,
    b.crp_country
  FROM cdp_data AS a
  LEFT JOIN crp_data AS b
    ON a.researcher_id = b.researcher_id
)
WHERE
  cdp_id = TRUE
  AND crp_id = TRUE
  AND cdp_given = TRUE
  AND crp_given = TRUE
  AND journal_country != crp_country
GROUP BY 1, 2
ORDER BY cnt DESC;


Query is running:   0%|          |

Downloading:   0%|          |

,journal_country,crp_country,cnt
0,Türkiye,Turkey,75149
1,The Netherlands,Netherlands,42319
2,Viet Nam,Vietnam,10143
3,Hong Kong,China,9040
4,United States,China,5364
...,...,...,...
7496,Indonesia,Oman,1
7497,Morocco,Malaysia,1
7498,United Kingdom,Fiji,1
7499,Oman,Algeria,1


## 3. ROR MAPPING DATA ( NOT RELIABLE FOR COUNTRY ENRICHMENT)

This query evaluates the impact of country enrichment by comparing the original CDP country coverage with the final coverage after enrichment, using the enriched audience table by ror mapping table. (Find the enriched table in institution enrichment ROR DATA step)

In [ ]:
%%bigquery  --project $pid


WITH stats AS (
  SELECT
    COUNT(*) AS total_target_rows,

    COUNTIF(country IS NOT NULL AND TRIM(country) != '') AS original_filled_rows,

    COUNTIF(enriched_country IS NOT NULL AND TRIM(enriched_country) != '') AS final_enriched_rows,


    COUNTIF((country IS NULL OR TRIM(country) = '') AND enriched_country IS NOT NULL) AS net_new_country_rows

FROM `dat-analytics-eng-ec869189.data_enrichment.audience_builder_big_enriched`  WHERE mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE
)

SELECT
  *,
  ROUND(original_filled_rows / total_target_rows * 100, 2) AS initial_fill_rate_percent,
  ROUND(final_enriched_rows / total_target_rows * 100, 2) AS final_fill_rate_percent,
  (final_enriched_rows - original_filled_rows) AS actual_rows_recovered
FROM stats

Query is running:   0%|          |

Downloading:   0%|          |

,total_target_rows,original_filled_rows,final_enriched_rows,net_new_country_rows,initial_fill_rate_percent,final_fill_rate_percent,actual_rows_recovered
0,17455879,16198402,16216964,18562,92.8,92.9,18562


Country data in CDP was already largely complete, and the enrichment provides only a small incremental improvement, adding country information to approximately 18.6K previously empty records. However, this enrichment should be interpreted with caution, as the country values are derived from ROR institution metadata rather than the authors’ actual countries, and therefore may not reliably represent individual author locations.

## 4. GA4 Export from funnel.io for Country Enrichment



*   The GA4 data is exported from Funnel.io into BigQuery.

The table is named >
dat-analytics-eng-ec869189.funnel_exports.ga4_data_for_enrichment
*  The table contains Google Analytics 4 event-level data, including identifiers (snid, client_id) , submission IDs, event counts, date, and geo related fields such as country and country ISO code.

*  The dataset includes data from February 20, 2024 up to today and is stored as a partitioned table.



###QUERY 1: Overlap Analysis - Enrcihment Potential

In [3]:
%%bigquery  --project $pid

  WITH
-- 1) Targeted B2C audience (filters MUST stay)
b2c_target AS (
  SELECT
    REGEXP_REPLACE(snid, r'_SN$', '') AS snid,
    NULLIF(TRIM(country), '') AS b2c_country
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
    OR mkt_pref_opt_in = TRUE
),

-- 2) Funnel GA4 data (clean country)
funnel_raw AS (
  SELECT
    snid___GA4__Google_Analytics AS snid,
    NULLIF(TRIM(Country___GA4__Google_Analytics), '') AS funnel_country,
    Date AS event_date
  FROM `dat-analytics-eng-ec869189.funnel_exports.ga4_data_for_enrichment`
  WHERE snid___GA4__Google_Analytics IS NOT NULL
),

-- 3) One country per SNID from Funnel
-- Strategy: most recent non-empty country
funnel_country_per_snid AS (
  SELECT
    snid,
    funnel_country
  FROM (
    SELECT
      snid,
      funnel_country,
      event_date,
      ROW_NUMBER() OVER (
        PARTITION BY snid
        ORDER BY event_date DESC
      ) AS rn
    FROM funnel_raw
    WHERE funnel_country IS NOT NULL
  )
  WHERE rn = 1
),

-- 4) Overlap set (this is your ~603k)
overlap AS (
  SELECT
    b.snid,
    b.b2c_country,
    f.funnel_country
  FROM b2c_target b
  JOIN funnel_country_per_snid f
    ON b.snid = f.snid
)

-- 5) FINAL METRICS
SELECT
  COUNT(DISTINCT snid) AS total_overlap_snids,

  -- B2C country coverage
  COUNT(DISTINCT IF(b2c_country IS NOT NULL, snid, NULL)) AS b2c_country_filled,
  COUNT(DISTINCT IF(b2c_country IS NULL, snid, NULL)) AS b2c_country_missing,

  -- Enrichment potential
  COUNT(DISTINCT IF(b2c_country IS NULL AND funnel_country IS NOT NULL, snid, NULL))
    AS enrichable_from_ga4,

  -- Still missing after GA4
  COUNT(DISTINCT IF(b2c_country IS NULL AND funnel_country IS NULL, snid, NULL))
    AS not_enrichable
FROM overlap;



Query is running:   0%|          |

Downloading:   0%|          |

,total_overlap_snids,b2c_country_filled,b2c_country_missing,enrichable_from_ga4,not_enrichable
0,2389057,1775265,613830,613830,0


###QUERY 2: Consistency check between datasets

In [6]:
%%bigquery  --project $pid

WITH
b2c_target AS (
  SELECT
    REGEXP_REPLACE(snid, r'_SN$', '') AS snid,
    NULLIF(TRIM(country), '') AS b2c_country
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
     OR mkt_pref_opt_in = TRUE
),
funnel_raw AS (
  SELECT
    snid___GA4__Google_Analytics AS snid,
    NULLIF(TRIM(Country___GA4__Google_Analytics), '') AS ga4_country,
    Date AS event_date
  FROM `dat-analytics-eng-ec869189.funnel_exports.ga4_data_for_enrichment`
),
funnel_country_per_snid AS (
  SELECT snid, ga4_country
  FROM (
    SELECT
      snid,
      ga4_country,
      event_date,
      ROW_NUMBER() OVER (
        PARTITION BY snid
        ORDER BY event_date DESC
      ) AS rn
    FROM funnel_raw
    WHERE ga4_country IS NOT NULL
  )
  WHERE rn = 1
),
both_filled AS (
  SELECT
    b.snid,
    b.b2c_country,
    f.ga4_country
  FROM b2c_target b
  JOIN funnel_country_per_snid f
    ON b.snid = f.snid
  WHERE b.b2c_country IS NOT NULL
    AND f.ga4_country IS NOT NULL
)

SELECT
  COUNT(DISTINCT snid) AS total_snids,
  COUNT(DISTINCT IF(ga4_country = b2c_country, snid, NULL)) AS matching_snids,
  COUNT(DISTINCT IF(ga4_country != b2c_country, snid, NULL)) AS mismatching_snids
FROM both_filled;


Query is running:   0%|          |

Downloading:   0%|          |

,total_snids,matching_snids,mismatching_snids
0,1775265,1482305,292963


###QUERY 3: Mismatching SNIDs

In [5]:
%%bigquery  --project $pid

WITH
b2c_target AS (
  SELECT
    REGEXP_REPLACE(snid, r'_SN$', '') AS snid,
    NULLIF(TRIM(country), '') AS b2c_country
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
     OR mkt_pref_opt_in = TRUE
),
funnel_raw AS (
  SELECT
    snid___GA4__Google_Analytics AS snid,
    NULLIF(TRIM(Country___GA4__Google_Analytics), '') AS ga4_country,
    Date AS event_date
  FROM `dat-analytics-eng-ec869189.funnel_exports.ga4_data_for_enrichment`
  WHERE snid___GA4__Google_Analytics IS NOT NULL
),
funnel_country_per_snid AS (
  SELECT snid, ga4_country
  FROM (
    SELECT
      snid,
      ga4_country,
      event_date,
      ROW_NUMBER() OVER (
        PARTITION BY snid
        ORDER BY event_date DESC
      ) AS rn
    FROM funnel_raw
    WHERE ga4_country IS NOT NULL
  )
  WHERE rn = 1
),
both_filled AS (
  SELECT
    b.snid,
    f.ga4_country,
    b.b2c_country
  FROM b2c_target b
  JOIN funnel_country_per_snid f
    ON b.snid = f.snid
  WHERE b.b2c_country IS NOT NULL
    AND f.ga4_country IS NOT NULL
)

SELECT
  snid,
  ga4_country,
  b2c_country
FROM both_filled
WHERE ga4_country != b2c_country
ORDER BY ga4_country, b2c_country
LIMIT 1000;


Query is running:   0%|          |

Downloading:   0%|          |

,snid,ga4_country,b2c_country
0,22b58633-00e2-4d9d-aa34-dfed6a2ca0f7,Afghanistan,Austria
1,b1a5b8f2-3360-4881-9973-5821f53a0efd,Afghanistan,Bahrain
2,919899ae-d02a-4217-9e5f-36ac9684a7f1,Afghanistan,Bahrain
3,fc868abb-9041-4761-a706-c7a86827d6fa,Afghanistan,Bahrain
4,905ea063-8c68-4afe-99e4-b023e8e3a566,Afghanistan,Belgium
...,...,...,...
995,bbeecfb9-2c35-4be0-8a82-3b2373780383,Andorra,France
996,fb7058f7-cf61-4fdd-a259-941dc45b4c82,Andorra,France
997,1033347f-833c-47d1-b6a6-18efad6927c0,Andorra,France
998,2bf7f64a-1d62-4711-9686-ec98520a1d3d,Andorra,France


###QUERY 4: Comparison of Countries in CDP & CRP & GA4 -> To see if Ga4 country is reliable

In [4]:
%%bigquery  --project $pid

WITH
b2c_target AS (
  SELECT
    REGEXP_REPLACE(snid, r'_SN$', '') AS snid_clean,
    NULLIF(TRIM(country), '') AS b2c_country
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = TRUE
     OR mkt_pref_opt_in = TRUE
),

funnel_raw AS (
  SELECT
    snid___GA4__Google_Analytics AS snid_clean,
    NULLIF(TRIM(Country___GA4__Google_Analytics), '') AS ga4_country,
    Date AS event_date
  FROM `dat-analytics-eng-ec869189.funnel_exports.ga4_data_for_enrichment`
  WHERE snid___GA4__Google_Analytics IS NOT NULL
),

funnel_country_per_snid AS (
  SELECT snid_clean, ga4_country
  FROM (
    SELECT
      snid_clean,
      ga4_country,
      event_date,
      ROW_NUMBER() OVER (
        PARTITION BY snid_clean
        ORDER BY event_date DESC
      ) AS rn
    FROM funnel_raw
    WHERE ga4_country IS NOT NULL
  )
  WHERE rn = 1
),

mismatching AS (
  SELECT
    b.snid_clean,
    f.ga4_country,
    b.b2c_country
  FROM b2c_target b
  JOIN funnel_country_per_snid f
    ON b.snid_clean = f.snid_clean
  WHERE b.b2c_country IS NOT NULL
    AND f.ga4_country IS NOT NULL
    AND b.b2c_country != f.ga4_country
),

crp_data AS (
  SELECT
    REGEXP_REPLACE(s.snid, r'_SN$', '') AS snid_clean,
    current_grid_country AS crp_country
  FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`,
       UNNEST(snids) AS s
  WHERE current_grid_country IS NOT NULL
),

final_set AS (
  SELECT
    m.snid_clean,
    m.ga4_country,
    m.b2c_country,
    c.crp_country
  FROM mismatching m
  JOIN crp_data c
    ON m.snid_clean = c.snid_clean
)

SELECT
  COUNT(*) AS total_records,

  -- 1️⃣ All three equal
  COUNTIF(
    ga4_country = b2c_country
    AND ga4_country = crp_country
  ) AS all_three_match,

  -- 2️⃣ CDP = CRP (GA4 different)
  COUNTIF(
    b2c_country = crp_country
    AND b2c_country != ga4_country
  ) AS cdp_equals_crp,

  -- 3️⃣ CRP = GA4 (CDP different)
  COUNTIF(
    crp_country = ga4_country
    AND crp_country != b2c_country
  ) AS crp_equals_ga4,

  -- 4️⃣ GA4 = CDP (CRP different)
  COUNTIF(
    ga4_country = b2c_country
    AND ga4_country != crp_country
  ) AS ga4_equals_cdp,

  -- 5️⃣ All three different
  COUNTIF(
    ga4_country != b2c_country
    AND ga4_country != crp_country
    AND b2c_country != crp_country
  ) AS all_three_different

FROM final_set;


Query is running:   0%|          |

Downloading:   0%|          |

,total_records,all_three_match,cdp_equals_crp,crp_equals_ga4,ga4_equals_cdp,all_three_different
0,144554,0,114046,22326,0,8182


* GA4 data might not be reliable as we thought

# **INSTITUTION ENRICHMENT**

# Enrichment analysis using ROR MAPPING TABLE

##1. AUTHORS DATA

How many CDP records with missing institution information can be matched to the Authors dataset using email, and

How many of those matched records can be successfully standardized using the ROR mapping table (i.e. assigned a valid ROR ID and standardized institution name).

###QUERY A

In [ ]:
%%bigquery  --project $pid


WITH cdp_empty_emails AS (

  --empty institution but available emails ~15M
  SELECT DISTINCT email
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE (mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE)
    AND (institution_name IS NULL OR TRIM(institution_name) = '')
    AND email IS NOT NULL AND email != ''
),

authors_mapping AS (
  -- Institution names from authors
  SELECT
    author_email,
    ARRAY_AGG(TRIM(institute) ORDER BY institute IS NOT NULL DESC LIMIT 1)[OFFSET(0)] AS raw_institute_name
  FROM `datasn-submissions-55677263.prod_joint_submissions_core.authors`
  WHERE institute IS NOT NULL AND TRIM(institute) != ''
  GROUP BY 1
),

final_enrichment_link AS (
  --CDP -> Authors -> ROR Master Table
  SELECT
    c.email,
    a.raw_institute_name,
    m.ror_id,
    m.ror_standardized_name
  FROM cdp_empty_emails c
  INNER JOIN authors_mapping a ON c.email = a.author_email
  LEFT JOIN `dat-analytics-eng-ec869189.data_enrichment.institution_ror_mapping_table` m
    ON LOWER(a.raw_institute_name) = LOWER(m.inst_name)
)

SELECT
  COUNT(*) as potential_email_matches,
  COUNT(ror_id) as finalized_standardized_rows,
  ROUND(COUNT(ror_id) / COUNT(*) * 100, 2) as standardization_rate
FROM final_enrichment_link


Query is running:   0%|          |

Downloading:   0%|          |

,potential_email_matches,finalized_standardized_rows,standardization_rate
0,30661385,25447222,82.99


Using Authors data, approximately 30.7M CDP records with missing institution information can be enriched.
Of these, 25.4M records (≈83%) can be standardized using the ROR mapping table, making them suitable for production use.
 (Note: The standardized count includes all records with any available ROR match, regardless of confidence score, including low-confidence matches.)

### QUERY B

This query evaluates the current and projected institution coverage in the CDP dataset for opt-in and legitimate-interest users.

It first calculates the baseline coverage, including:

the total number of target CDP records, and

how many of those records already have an institution name populated.

It then estimates the enrichment impact by identifying CDP records with missing institution information that can be enriched via the Authors dataset and successfully standardized using the ROR mapping table.

Finally, the query compares the initial institution fill rate with the expected fill rate after enrichment, showing how much overall coverage would improve if the Authors→ROR enrichment were applied.


In [ ]:
%%bigquery  --project $pid

WITH cdp_base AS (

  SELECT
    COUNT(*) AS total_target_rows,
    COUNTIF(institution_name IS NOT NULL AND TRIM(institution_name) != '') AS already_filled_rows
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE
),

authors_with_ror AS (

  SELECT
    auth.author_email,
    ANY_VALUE(m.ror_id) as valid_ror_id
  FROM `datasn-submissions-55677263.prod_joint_submissions_core.authors` AS auth
  INNER JOIN `dat-analytics-eng-ec869189.data_enrichment.institution_ror_mapping_table` AS m
    ON LOWER(TRIM(auth.institute)) = LOWER(TRIM(m.inst_name))
  GROUP BY 1
),

enrichment_results AS (

  SELECT
    COUNTIF(
      (cdp.institution_name IS NULL OR TRIM(cdp.institution_name) = '')
      AND ar.valid_ror_id IS NOT NULL
    ) AS new_enriched_rows
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big` AS cdp
  LEFT JOIN authors_with_ror ar ON cdp.email = ar.author_email
  WHERE cdp.mkt_pref_opt_in = TRUE OR cdp.is_legitimate_interest_author_opted_in = TRUE
)

SELECT
  base.total_target_rows,
  base.already_filled_rows,
  enr.new_enriched_rows,
  ROUND(base.already_filled_rows / base.total_target_rows * 100, 2) AS initial_fill_rate_percent,
  (base.already_filled_rows + enr.new_enriched_rows) AS final_filled_total,
  ROUND((base.already_filled_rows + enr.new_enriched_rows) / base.total_target_rows * 100, 2) AS final_fill_rate_percent
FROM cdp_base base, enrichment_results enr


Query is running:   0%|          |

Downloading:   0%|          |

,total_target_rows,already_filled_rows,new_enriched_rows,initial_fill_rate_percent,final_filled_total,final_fill_rate_percent
0,15806904,126509,9481776,0.8,9608285,60.79


## 2. CRP DATA

## QUERY A

This query calculates the expected increase in standardized institution coverage for legitimate interest CDP users by enriching missing institution data from CRP profiles (via Researcher ID and SNID) and validating those enrichments through the ROR mapping table.

In [ ]:
%%bigquery  --project $pid


WITH cdp_base AS (
  SELECT
    COUNT(*) AS total_target_rows,
    COUNTIF(institution_name IS NOT NULL AND TRIM(institution_name) != '') AS already_filled_rows
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big`
  WHERE is_legitimate_interest_author_opted_in = true
),

crp_source AS (
  SELECT researcher_id, NULL as snid, current_institute_name as inst_name FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out` WHERE researcher_id IS NOT NULL
  UNION ALL
  SELECT NULL as researcher_id, s.snid, current_institute_name as inst_name FROM `aceproducts-143febd8.core_researcher_profiles.researcher_profiles_out`, UNNEST(snids) AS s
),

cdp_crp_match AS (
  SELECT DISTINCT
    cdp.researcher_id,
    cdp.snid,
    cs.inst_name as raw_crp_inst_name
  FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big` AS cdp
  INNER JOIN crp_source AS cs
    ON (cdp.researcher_id = cs.researcher_id OR cdp.snid = cs.snid)
  WHERE cdp.is_legitimate_interest_author_opted_in = true
    AND (cdp.institution_name IS NULL OR TRIM(cdp.institution_name) = '')
    AND cs.inst_name IS NOT NULL
),

final_standardized_matches AS (
  SELECT
    COUNT(DISTINCT COALESCE(CAST(m.researcher_id AS STRING), CAST(m.snid AS STRING))) AS new_enriched_rows
  FROM cdp_crp_match m
  INNER JOIN `dat-analytics-eng-ec869189.data_enrichment.institution_ror_mapping_table` ror
    ON LOWER(TRIM(m.raw_crp_inst_name)) = LOWER(TRIM(ror.inst_name))
)

-- 5. Final
SELECT
  base.total_target_rows,
  base.already_filled_rows,
  enr.new_enriched_rows,
  ROUND(base.already_filled_rows / base.total_target_rows * 100, 2) AS initial_fill_rate_percent,
  (base.already_filled_rows + enr.new_enriched_rows) AS final_filled_total,
  ROUND((base.already_filled_rows + enr.new_enriched_rows) / base.total_target_rows * 100, 2) AS final_fill_rate_percent
FROM cdp_base base, final_standardized_matches enr

------------------------------------------------------------------


Query is running:   0%|          |

Downloading:   0%|          |

,total_target_rows,already_filled_rows,new_enriched_rows,initial_fill_rate_percent,final_filled_total,final_fill_rate_percent
0,14093253,125097,2599369,0.89,2724466,19.33


- Using CRP profile data, institution enrichment for legitimate interest users increases coverage from 0.9% to 19.3%, adding standardized institution information for approximately 2.6M records. However, the institution coverage obtained from CRP may partially overlap with Authors-based enrichment

## Creating an Enriched Dataset using available Institutions in Authors and CRP data

This query creates an enriched version of the CDP audience table by filling missing institution and country information using ROR-standardized metadata. The enrichment process follows a strict hierarchy to ensure data accuracy and recency:

Priority & Source: It first matches records with Authors data (via email). If no match is found, it proceeds to CRP data (via Researcher ID or SNID), exclusively for users with legitimate interest (LI) status.

Recency Logic: This query specifically selects the most recent institution based on the latest submission date available in both the Submissions and CRP datasets.

It adds the following new columns on top of all existing CDP columns (cdp.*):

enriched_institution_name

The final institution name after enrichment.

Logic: Prioritizes the institution from the most recent submission in Authors data. If missing, it uses the most recent record from CRP (only for LI users).

ror_id

The unique ROR identifier for the enriched institution.

inst_confidence_score

The confidence score of the ROR mapping process.

Note: This is provided for visibility into the mapping quality; the enrichment logic itself prioritizes record recency (latest submission) rather than the score.

enriched_country

The country derived from ROR metadata, filled only if the original CDP country field is empty.

In [ ]:
CREATE OR REPLACE TABLE `dat-analytics-eng-ec869189.data_enrichment.audience_builder_big_enriched` AS

WITH
/* 1) Authors tablosundan kurum VE ülke bilgisini al */
latest_author_data AS (
  SELECT
    LOWER(TRIM(a.author_email)) AS author_email,
    NULLIF(TRIM(a.institute), '') AS raw_author_institute,
    NULLIF(TRIM(a.author_country), '') AS raw_author_country, -- Authors tablosundaki ülke
    s.submission_date
  FROM `datasn-submissions-55677263.prod_joint_submissions_core.authors` a
  INNER JOIN `datasn-submissions-55677263.prod_joint_submissions_core.submissions` s
    ON a.submission_id = s.submission_id
  WHERE a.author_email IS NOT NULL
    AND a.institute IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY LOWER(TRIM(a.author_email))
    ORDER BY s.submission_date DESC, a.submission_id DESC
  ) = 1
),

/* 2) Eşleşmeyi hem kurum hem ülke üzerinden yap */
author_ror_matches AS (
  SELECT
    lad.author_email,
    m.ror_standardized_name,
    m.ror_country,
    m.confidence_score
  FROM latest_author_data lad
  INNER JOIN `dat-analytics-eng-ec869189.data_enrichment.institution_ror_mapping_table` AS m
    ON LOWER(TRIM(lad.raw_author_institute)) = LOWER(TRIM(m.inst_name))
    -- ÜLKE KONTROLÜ: Authors'taki ülke ile ROR tablosundaki ülke eşleşmeli
    AND LOWER(TRIM(lad.raw_author_country)) = LOWER(TRIM(m.ror_country))
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY lad.author_email
    ORDER BY m.confidence_score DESC
  ) = 1
)

/* 3) Ana tabloyu (CDP) zenginleştir */
SELECT
  cdp.*,
  CASE
    WHEN (cdp.institution_name IS NULL OR TRIM(cdp.institution_name) = '')
      THEN arm.ror_standardized_name
    ELSE cdp.institution_name
  END AS enriched_institution_name,

  -- Ülke bilgisini de zenginleştirelim
  CASE
    WHEN (cdp.country IS NULL OR TRIM(cdp.country) = '')
      THEN arm.ror_country
    ELSE cdp.country
  END AS enriched_country

FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big` AS cdp
LEFT JOIN author_ror_matches arm
  ON LOWER(TRIM(cdp.email)) = arm.author_email;

Query is running:   0%|          |

""


TESTING ROR MAPPING

The primary goal of this query is to perform a morphological analysis of institution name discrepancies. It isolates high-confidence matches (confidence_score = 1) where the author's input differs from the ROR official name, then categorizes the "mismatch reason" based on the string length relationship.
Discrepancy Categorization (mismatch_reason):

Initials/Abbreviation: Raw input is shorter than the ROR name (e.g., "MIT" → "Massachusetts Institute of Technology").

Minor Variation: stylistic reconciliation of names that are already largely correct but vary in punctuation, word order, or casing.

Excessive Detail: Raw input is longer, often containing departments or extra addresses (e.g., "Harvard University, Dept of Physics" → "Harvard University").

Completely Different Name:A semantic resolution where the raw input and the standardized name share no common keywords (excluding words shorter than 3 characters).

In [ ]:
SELECT
  cdp.email,
  -- Raw institution name provided by authors
  arm.latest_author_raw_inst,
  arm.author_raw_country,

  -- Official standardized name from ROR mapping
  arm.enriched_ror_inst_name,
  arm.ror_country AS matched_ror_country,

  -- Confidence score
  arm.confidence_score,

  -- Fixed Mismatch Reason Analysis using REGEXP_CONTAINS
  CASE
    -- 1. Keyword Audit: Check if any word (3+ chars) from raw exists in standardized
    WHEN NOT EXISTS (
      SELECT 1
      FROM UNNEST(REGEXP_EXTRACT_ALL(LOWER(arm.latest_author_raw_inst), r'[a-z0-9]{3,}')) AS word
      WHERE REGEXP_CONTAINS(LOWER(arm.enriched_ror_inst_name), word)
    ) THEN 'Completely Different Name'

    -- 2. Categorization by length
    WHEN LENGTH(arm.latest_author_raw_inst) < LENGTH(arm.enriched_ror_inst_name) THEN 'Initials/Abbreviation'
    WHEN LENGTH(arm.latest_author_raw_inst) > LENGTH(arm.enriched_ror_inst_name) THEN 'Excessive Detail'
    ELSE 'Minor Variation'
  END AS mismatch_reason

FROM `dat-analytics-eng-ec869189.prod_audience_explorer_analytics_dbt_presentation.audience_builder_big` AS cdp
INNER JOIN (
  /* Subquery to consolidate Author and ROR data */
  WITH latest_author_data AS (
    SELECT
      LOWER(TRIM(a.author_email)) AS author_email,
      NULLIF(TRIM(a.institute), '') AS latest_author_raw_inst,
      NULLIF(TRIM(a.author_country), '') AS author_raw_country,
      s.submission_date
    FROM `datasn-submissions-55677263.prod_joint_submissions_core.authors` a
    INNER JOIN `datasn-submissions-55677263.prod_joint_submissions_core.submissions` s
      ON a.submission_id = s.submission_id
    QUALIFY ROW_NUMBER() OVER (
      PARTITION BY LOWER(TRIM(a.author_email))
      ORDER BY s.submission_date DESC, a.submission_id DESC
    ) = 1
  )
  SELECT
    lad.author_email,
    lad.latest_author_raw_inst,
    lad.author_raw_country,
    m.ror_standardized_name AS enriched_ror_inst_name,
    m.ror_country,
    m.confidence_score
  FROM latest_author_data lad
  INNER JOIN `dat-analytics-eng-ec869189.data_enrichment.institution_ror_mapping_table` AS m
    ON LOWER(TRIM(lad.latest_author_raw_inst)) = LOWER(TRIM(m.inst_name))
  WHERE m.confidence_score = 1
) arm
  ON LOWER(TRIM(cdp.email)) = arm.author_email

/* Isolate records where the raw input does not match the standardized name */
WHERE LOWER(TRIM(arm.latest_author_raw_inst)) != LOWER(TRIM(arm.enriched_ror_inst_name))

ORDER BY mismatch_reason DESC, LENGTH(arm.latest_author_raw_inst) ASC;

we should normalize ror mapping table because there are false postives in names with 1.0 confidence scores

## Enriched Table Analysis : Before and After

###QUERY A

In [ ]:
%%bigquery  --project $pid

select
  COUNT(*) AS total_target_rows,


  COUNTIF(institution_name IS NOT NULL AND TRIM(institution_name) != '') AS original_filled_rows,


  COUNTIF(enriched_institution_name IS NOT NULL AND TRIM(enriched_institution_name) != '') AS final_enriched_rows,


  ROUND(
    COUNTIF(institution_name IS NOT NULL AND TRIM(institution_name) != '') / COUNT(*) * 100, 2
  ) AS initial_fill_rate_percent,


  ROUND(
    COUNTIF(enriched_institution_name IS NOT NULL AND TRIM(enriched_institution_name) != '') / COUNT(*) * 100, 2
  ) AS final_fill_rate_percent,

  (COUNTIF(enriched_institution_name IS NOT NULL AND TRIM(enriched_institution_name) != '') -
   COUNTIF(institution_name IS NOT NULL AND TRIM(institution_name) != '')) AS net_new_rows_recovered

FROM `dat-analytics-eng-ec869189.data_enrichment.audience_builder_big_enriched`
WHERE mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE


Query is running:   0%|          |

Downloading:   0%|          |

,total_target_rows,original_filled_rows,final_enriched_rows,initial_fill_rate_percent,final_fill_rate_percent,net_new_rows_recovered
0,17455879,168277,11571137,0.96,66.29,11402860




*   Out of 17.46M target records, only 168K (0.96%) originally had an institution populated. After applying the Authors + CRP enrichment with ROR standardization, 11.57M records now have a populated institution, increasing overall coverage to 66.29%
*   In net terms, the enrichment process recovered institution data for approximately 11.4M previously empty records





###QUERY B: Low-confidence institution match

This query counts how many enriched CDP records have a low-confidence institution match below 0.89

In [ ]:
%%bigquery  --project $pid

SELECT
  COUNT(*) as low_confidence_count,
FROM `dat-analytics-eng-ec869189.data_enrichment.audience_builder_big_enriched`
WHERE inst_confidence_score < 0.89
  AND (mkt_pref_opt_in = TRUE OR is_legitimate_interest_author_opted_in = TRUE)



Query is running:   0%|          |

Downloading:   0%|          |

,low_confidence_count
0,1400969


~1.4M - they may be less reliable for high-precision use cases